# NYC Taxi Demand Forecasting — Feature Engineering

This notebook transforms the validated 2025 zone-hour demand panel into a modeling-ready dataset by constructing calendar and historical demand features for one-hour-ahead forecasting.

## Objective

1. Load and validate the processed hourly demand panel.

2. Construct calendar features representing recurring temporal demand patterns.

3. Create lagged and rolling demand features using only information available before each prediction hour.

4. Validate feature construction to prevent target leakage.

5. Save the completed modeling dataset for downstream forecasting.

## 1. Imports

Load the Python libraries used for feature engineering and validation.

In [32]:
import pandas as pd
from pathlib import Path
from pandas.tseries.holiday import USFederalHolidayCalendar

## 2. Load Processed Hourly Demand

Load the balanced zone-hour demand panel constructed in the historical demand notebook. This dataset contains one observation for every hour of 2025 across the 174 selected pickup zones.

In [33]:
processed_dir = Path("../data/processed")
demand_path = processed_dir / "yellow_taxi_hourly_demand_2025.parquet"

demand = pd.read_parquet(demand_path)

demand.head()

,pickup_hour,PULocationID,trip_count
0,2025-01-01,237,261
1,2025-01-01,161,274
2,2025-01-01,132,233
3,2025-01-01,236,128
4,2025-01-01,186,155


In [34]:
print("Shape:", demand.shape)
print("Date range:", demand["pickup_hour"].min(), "to", demand["pickup_hour"].max())
print("Zones:", demand["PULocationID"].nunique())
print("Missing values:", demand.isna().sum().sum())

Shape: (1524240, 3)
Date range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
Zones: 174
Missing values: 0


## 3. Sort and Validate Chronological Structure

Sort observations by pickup zone and timestamp before constructing historical demand features. Correct chronological ordering ensures that lagged and rolling features reference earlier observations from the same zone.

In [35]:
demand = (
    demand
    .sort_values(["PULocationID", "pickup_hour"])
    .reset_index(drop=True)
)

demand.head(10)

,pickup_hour,PULocationID,trip_count
0,2025-01-01 00:00:00,4,28
1,2025-01-01 01:00:00,4,54
2,2025-01-01 02:00:00,4,52
3,2025-01-01 03:00:00,4,42
4,2025-01-01 04:00:00,4,32
5,2025-01-01 05:00:00,4,5
6,2025-01-01 06:00:00,4,5
7,2025-01-01 07:00:00,4,3
8,2025-01-01 08:00:00,4,1
9,2025-01-01 09:00:00,4,2


In [36]:
zone_hours = demand.groupby("PULocationID").size()

print("Minimum hours per zone:", zone_hours.min())
print("Maximum hours per zone:", zone_hours.max())
print("Expected hours per zone:", 8760)

Minimum hours per zone: 8760
Maximum hours per zone: 8760
Expected hours per zone: 8760


## 4. Create Calendar Features

Extract calendar information from each prediction timestamp to represent the recurring hourly, weekly, and seasonal demand patterns identified during historical demand analysis.

In [37]:
demand["hour"] = demand["pickup_hour"].dt.hour
demand["day_of_week"] = demand["pickup_hour"].dt.dayofweek
demand["month"] = demand["pickup_hour"].dt.month
demand["day_of_month"] = demand["pickup_hour"].dt.day
demand["is_weekend"] = (demand["day_of_week"] >= 5).astype(int)

demand.head()

,pickup_hour,PULocationID,trip_count,hour,day_of_week,month,day_of_month,is_weekend
0,2025-01-01 00:00:00,4,28,0,2,1,1,0
1,2025-01-01 01:00:00,4,54,1,2,1,1,0
2,2025-01-01 02:00:00,4,52,2,2,1,1,0
3,2025-01-01 03:00:00,4,42,3,2,1,1,0
4,2025-01-01 04:00:00,4,32,4,2,1,1,0


In [38]:
demand[
    ["pickup_hour", "hour", "day_of_week", "month", "day_of_month", "is_weekend"]
].head(24)

,pickup_hour,hour,day_of_week,month,day_of_month,is_weekend
0,2025-01-01 00:00:00,0,2,1,1,0
1,2025-01-01 01:00:00,1,2,1,1,0
2,2025-01-01 02:00:00,2,2,1,1,0
3,2025-01-01 03:00:00,3,2,1,1,0
4,2025-01-01 04:00:00,4,2,1,1,0
5,2025-01-01 05:00:00,5,2,1,1,0
6,2025-01-01 06:00:00,6,2,1,1,0
7,2025-01-01 07:00:00,7,2,1,1,0
8,2025-01-01 08:00:00,8,2,1,1,0
9,2025-01-01 09:00:00,9,2,1,1,0


## 5. Create Lagged Demand Features

Create historical demand features that represent pickup volume in the same zone at earlier points in time. These features capture short-term persistence as well as recurring daily and weekly demand patterns.

Each lag is calculated independently within a pickup zone so that demand from one zone is never used as the historical demand of another.

In [39]:
zone_groups = demand.groupby("PULocationID")["trip_count"]

demand["lag_1"] = zone_groups.shift(1)
demand["lag_24"] = zone_groups.shift(24)
demand["lag_168"] = zone_groups.shift(168)

In [40]:
example_zone = demand["PULocationID"].iloc[0]

demand.loc[
    demand["PULocationID"] == example_zone,
    ["pickup_hour", "trip_count", "lag_1", "lag_24", "lag_168"]
].head(30)

,pickup_hour,trip_count,lag_1,lag_24,lag_168
0,2025-01-01 00:00:00,28,NaN,NaN,NaN
1,2025-01-01 01:00:00,54,28.0,NaN,NaN
2,2025-01-01 02:00:00,52,54.0,NaN,NaN
3,2025-01-01 03:00:00,42,52.0,NaN,NaN
4,2025-01-01 04:00:00,32,42.0,NaN,NaN
5,2025-01-01 05:00:00,5,32.0,NaN,NaN
6,2025-01-01 06:00:00,5,5.0,NaN,NaN
7,2025-01-01 07:00:00,3,5.0,NaN,NaN
8,2025-01-01 08:00:00,1,3.0,NaN,NaN
9,2025-01-01 09:00:00,2,1.0,NaN,NaN


In [41]:
print(demand[["lag_1", "lag_24", "lag_168"]].isna().sum())

lag_1        174
lag_24      4176
lag_168    29232
dtype: int64


## 6. Create Rolling Demand Features

Create rolling historical demand features that summarize recent pickup activity within each zone. Rolling averages provide a smoother representation of recent demand than individual lag values and can help capture short-term changes in activity.

Each rolling calculation is shifted by one hour before aggregation so that the current hour's demand is never included in its own predictors.

In [42]:
demand["rolling_mean_24"] = (
    demand.groupby("PULocationID")["trip_count"]
    .transform(lambda x: x.shift(1).rolling(24).mean())
)

demand["rolling_mean_168"] = (
    demand.groupby("PULocationID")["trip_count"]
    .transform(lambda x: x.shift(1).rolling(168).mean())
)

In [43]:
print(
    demand[
        ["rolling_mean_24", "rolling_mean_168"]
    ].isna().sum()
)

rolling_mean_24      4176
rolling_mean_168    29232
dtype: int64


In [44]:
demand.loc[
    demand["PULocationID"] == example_zone,
    [
        "pickup_hour",
        "trip_count",
        "lag_1",
        "lag_24",
        "lag_168",
        "rolling_mean_24",
        "rolling_mean_168"
    ]
].iloc[165:175]

,pickup_hour,trip_count,lag_1,lag_24,lag_168,rolling_mean_24,rolling_mean_168
165,2025-01-07 21:00:00,4,0.0,1.0,NaN,2.541667,NaN
166,2025-01-07 22:00:00,2,4.0,2.0,NaN,2.666667,NaN
167,2025-01-07 23:00:00,0,2.0,4.0,NaN,2.666667,NaN
168,2025-01-08 00:00:00,3,0.0,0.0,28.0,2.500000,4.988095
169,2025-01-08 01:00:00,3,3.0,1.0,54.0,2.625000,4.839286
170,2025-01-08 02:00:00,2,3.0,1.0,52.0,2.708333,4.535714
171,2025-01-08 03:00:00,0,2.0,0.0,42.0,2.750000,4.238095
172,2025-01-08 04:00:00,0,0.0,0.0,32.0,2.750000,3.988095
173,2025-01-08 05:00:00,2,0.0,0.0,5.0,2.750000,3.797619
174,2025-01-08 06:00:00,9,2.0,5.0,5.0,2.833333,3.779762


## 7. Add Holiday Features

Add an indicator for U.S. federal holidays to capture predictable changes in travel behavior around major calendar events. Unlike historical demand features, holiday status is known in advance and can therefore be used without introducing target leakage.

In [45]:
calendar = USFederalHolidayCalendar()

holidays = calendar.holidays(
    start=demand["pickup_hour"].min().normalize(),
    end=demand["pickup_hour"].max().normalize()
)

demand["is_holiday"] = (
    demand["pickup_hour"].dt.normalize().isin(holidays)
).astype(int)

holidays

DatetimeIndex(['2025-01-01', '2025-01-20', '2025-02-17', '2025-05-26',
               '2025-06-19', '2025-07-04', '2025-09-01', '2025-10-13',
               '2025-11-11', '2025-11-27', '2025-12-25'],
              dtype='datetime64[ns]', freq=None)

In [46]:
demand.loc[
    demand["is_holiday"] == 1,
    ["pickup_hour", "is_holiday"]
].drop_duplicates().head(24)

,pickup_hour,is_holiday
0,2025-01-01 00:00:00,1
1,2025-01-01 01:00:00,1
2,2025-01-01 02:00:00,1
3,2025-01-01 03:00:00,1
4,2025-01-01 04:00:00,1
5,2025-01-01 05:00:00,1
6,2025-01-01 06:00:00,1
7,2025-01-01 07:00:00,1
8,2025-01-01 08:00:00,1
9,2025-01-01 09:00:00,1


## 8. Remove Rows Without Sufficient Historical Context

The first seven days of each zone do not contain enough prior observations to calculate the 168-hour lag and rolling features. Remove rows with incomplete historical features so every modeling observation contains the same predictor set.

In [47]:
historical_features = [
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_mean_168"
]

model_data = demand.dropna(subset=historical_features).copy()

In [48]:
print("Original rows:", f"{len(demand):,}")
print("Modeling rows:", f"{len(model_data):,}")
print("Rows removed:", f"{len(demand) - len(model_data):,}")
print("Remaining missing values:", model_data.isna().sum().sum())
print(
    "Modeling date range:",
    model_data["pickup_hour"].min(),
    "to",
    model_data["pickup_hour"].max()
)

Original rows: 1,524,240
Modeling rows: 1,495,008
Rows removed: 29,232
Remaining missing values: 0
Modeling date range: 2025-01-08 00:00:00 to 2025-12-31 23:00:00


## 9. Validate Historical Features

Verify that lagged demand features correspond to the intended historical observations and confirm that all demand-derived predictors use information strictly preceding the prediction timestamp.

In [49]:
validation_zone = model_data["PULocationID"].iloc[0]
validation_time = pd.Timestamp("2025-06-15 18:00:00")

validation_row = model_data.loc[
    (model_data["PULocationID"] == validation_zone) &
    (model_data["pickup_hour"] == validation_time)
]

validation_row[
    [
        "pickup_hour",
        "PULocationID",
        "trip_count",
        "lag_1",
        "lag_24",
        "lag_168",
        "rolling_mean_24",
        "rolling_mean_168"
    ]
]

,pickup_hour,PULocationID,trip_count,lag_1,lag_24,lag_168,rolling_mean_24,rolling_mean_168
3978,2025-06-15 18:00:00,4,18,17.0,30.0,28.0,32.083333,19.857143


In [50]:
validation_history = demand.loc[
    (demand["PULocationID"] == validation_zone) &
    (
        demand["pickup_hour"].isin([
            validation_time - pd.Timedelta(hours=1),
            validation_time - pd.Timedelta(hours=24),
            validation_time - pd.Timedelta(hours=168)
        ])
    ),
    ["pickup_hour", "trip_count"]
]

validation_history

,pickup_hour,trip_count
3810,2025-06-08 18:00:00,28
3954,2025-06-14 18:00:00,30
3977,2025-06-15 17:00:00,17


## 10. Review Final Feature Set

Inspect the completed feature-engineered dataset before saving it for model development.

In [51]:
model_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1495008 entries, 168 to 1524239
Data columns (total 14 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   pickup_hour       1495008 non-null  datetime64[ns]
 1   PULocationID      1495008 non-null  int32         
 2   trip_count        1495008 non-null  int64         
 3   hour              1495008 non-null  int32         
 4   day_of_week       1495008 non-null  int32         
 5   month             1495008 non-null  int32         
 6   day_of_month      1495008 non-null  int32         
 7   is_weekend        1495008 non-null  int64         
 8   lag_1             1495008 non-null  float64       
 9   lag_24            1495008 non-null  float64       
 10  lag_168           1495008 non-null  float64       
 11  rolling_mean_24   1495008 non-null  float64       
 12  rolling_mean_168  1495008 non-null  float64       
 13  is_holiday        1495008 non-null  int64    

In [52]:
model_data.head()

,pickup_hour,PULocationID,trip_count,hour,day_of_week,month,day_of_month,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_mean_168,is_holiday
168,2025-01-08 00:00:00,4,3,0,2,1,8,0,0.0,0.0,28.0,2.500000,4.988095,0
169,2025-01-08 01:00:00,4,3,1,2,1,8,0,3.0,1.0,54.0,2.625000,4.839286,0
170,2025-01-08 02:00:00,4,2,2,2,1,8,0,3.0,1.0,52.0,2.708333,4.535714,0
171,2025-01-08 03:00:00,4,0,3,2,1,8,0,2.0,0.0,42.0,2.750000,4.238095,0
172,2025-01-08 04:00:00,4,0,4,2,1,8,0,0.0,0.0,32.0,2.750000,3.988095,0


## 11. Save Feature-Engineered Dataset

Save the completed modeling dataset as a Parquet file for use in downstream model development and evaluation.

In [53]:
output_path = processed_dir / "yellow_taxi_modeling_features_2025.parquet"

model_data.to_parquet(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1_000_000:.1f} MB")

Saved to: ../data/processed/yellow_taxi_modeling_features_2025.parquet
File size: 14.3 MB


In [54]:
check = pd.read_parquet(output_path)

print("Shape:", check.shape)
print("Date range:", check["pickup_hour"].min(), "to", check["pickup_hour"].max())
print("Zones:", check["PULocationID"].nunique())
print("Missing values:", check.isna().sum().sum())

Shape: (1495008, 14)
Date range: 2025-01-08 00:00:00 to 2025-12-31 23:00:00
Zones: 174
Missing values: 0


## 12. Key Findings

**Observation:** The feature-engineering pipeline produced a complete modeling dataset containing calendar, holiday, lagged-demand, and rolling historical features for 174 pickup zones. After removing the seven-day warm-up period required by the longest historical features, 1,495,008 zone-hour observations remain with no missing values.

All demand-derived predictors were constructed independently within each pickup zone and use only observations preceding the prediction timestamp. Lag features capture recent, daily, and weekly demand relationships, while shifted rolling averages summarize recent demand without incorporating the current target. The resulting dataset is therefore suitable for chronological model development and evaluation without target leakage.